In [ ]:
!pip install qiskit qiskit-ibm-runtime qiskit-aer pennylane pennylane-qiskit tensorflow scikit-learn tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 

In [ ]:
# ── Cell 1: Imports ──────────────────────────────────────
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, amplitude_damping_error, ReadoutError
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.decomposition import PCA
import tensorflow as tf
import json

In [ ]:
# ── Cell 2: Paste your IBM token here ────────────────────
IBM_API_TOKEN = "PASTE_YOUR_TOKEN_HERE"
USE_REAL      = False   # keep False to use Aer simulator
                        # set True only for real IBM hardware

In [ ]:
# ── Cell 3: Config ───────────────────────────────────────
N_QUBITS     = 8
N_FEATURES   = 8
N_TRAIN      = 400
N_TEST       = 100
STEPS        = 100
LR           = 0.01
BATCH        = 20
NOISE_LEVELS = [0.0, 0.001, 0.005, 0.01, 0.02]
GVS_LO, GVS_HI = 1e-4, 1e-2
FMF_LO, FMF_HI = 0.3,  0.7
D_MIN, D_MAX    = 1, 5
K_ADAPT         = 10
W_WINDOW        = 10

In [ ]:
# ── Cell 4: Load data ────────────────────────────────────
(x_tr, y_tr), (x_te, y_te) = tf.keras.datasets.fashion_mnist.load_data()
x_tr = x_tr / 255.0
x_te = x_te / 255.0

# Filter classes 0 and 1
mask_tr = np.isin(y_tr, [0, 1])
mask_te = np.isin(y_te, [0, 1])
x_tr, y_tr = x_tr[mask_tr], y_tr[mask_tr]
x_te, y_te = x_te[mask_te], y_te[mask_te]

# PCA-8
x_tr = x_tr.reshape(len(x_tr), -1)
x_te = x_te.reshape(len(x_te), -1)
pca  = PCA(N_FEATURES)
x_tr = pca.fit_transform(x_tr)
x_te = pca.transform(x_te)

# Scale to [0, π]
mn, mx = x_tr.min(), x_tr.max()
x_tr = (x_tr - mn) * np.pi / (mx - mn)
x_te = (x_te - mn) * np.pi / (mx - mn)

# Binary labels
y_tr = np.array([1 if y == 0 else 0 for y in y_tr])
y_te = np.array([1 if y == 0 else 0 for y in y_te])

# Subsample
idx_tr = np.random.choice(len(x_tr), N_TRAIN, replace=False)
idx_te = np.random.choice(len(x_te), N_TEST,  replace=False)
X_train, Y_train = x_tr[idx_tr], y_tr[idx_tr]
X_test,  Y_test  = x_te[idx_te], y_te[idx_te]

print(f"Train: {len(X_train)}  Test: {len(X_test)}")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train: 400  Test: 100


In [ ]:
# ── Cell 5: Build noise model (ibm_brisbane profile) ─────
def build_noise_model(p_depol=0.001, p_damp=0.0005, readout_err=0.03):
    nm = NoiseModel()
    if p_depol > 0:
        e1q = depolarizing_error(p_depol / 4, 1)
        e2q = depolarizing_error(p_depol, 2)
        nm.add_all_qubit_quantum_error(e1q, ['rx','ry','rz','h','x'])
        nm.add_all_qubit_quantum_error(e2q, ['cx','cz'])
    if p_damp > 0:
        e_damp = amplitude_damping_error(p_damp)
        nm.add_all_qubit_quantum_error(e_damp, ['rx','ry','rz'])
    if readout_err > 0:
        ro = ReadoutError([[1-readout_err, readout_err],
                           [readout_err, 1-readout_err]])
        nm.add_all_qubit_readout_error(ro)
    return nm

In [ ]:
# ── Cell 6: Ansatz + circuit ──────────────────────────────
def U_SU4(params, wires):
    qml.U3(params[0], params[1], params[2], wires=wires[0])
    qml.U3(params[3], params[4], params[5], wires=wires[1])
    qml.CNOT(wires=[wires[0], wires[1]])
    qml.RY(params[6], wires=wires[0])
    qml.RZ(params[7], wires=wires[1])
    qml.CNOT(wires=[wires[1], wires[0]])
    qml.RY(params[8], wires=wires[0])
    qml.CNOT(wires=[wires[0], wires[1]])
    qml.U3(params[9],  params[10], params[11], wires=wires[0])
    qml.U3(params[12], params[13], params[14], wires=wires[1])

def Pooling(params, wires):
    qml.CRZ(params[0], wires=[wires[0], wires[1]])
    qml.PauliX(wires=wires[0])
    qml.CRX(params[1], wires=[wires[0], wires[1]])

def build_circuit(depth, noise_model=None, shots=1024):
    if noise_model is not None:
        backend = AerSimulator(noise_model=noise_model)
        dev = qml.device("qiskit.aer", wires=N_QUBITS,
                          backend=backend, shots=shots)
    else:
        dev = qml.device("default.qubit", wires=N_QUBITS)

    @qml.qnode(dev)
    def circuit(x, params):
        qml.AngleEmbedding(x, wires=range(N_QUBITS), rotation='Y')
        ptr    = 0
        active = list(range(N_QUBITS))
        for _ in range(depth):
            n = len(active)
            for i in range(n):
                w = [active[i], active[(i+1) % n]]
                U_SU4(params[ptr:ptr+15], wires=w);  ptr += 15
            new_active = []
            for i in range(0, len(active)-1, 2):
                Pooling(params[ptr:ptr+2],
                        wires=[active[i], active[i+1]]);  ptr += 2
                new_active.append(active[i+1])
            if len(active) % 2 == 1:
                new_active.append(active[-1])
            active = new_active
            if len(active) <= 1:
                break
        return qml.probs(wires=active[0])
    return circuit

def n_params(depth):
    total, n = 0, N_QUBITS
    for _ in range(depth):
        total += n * 15 + (n // 2) * 2
        n = max(1, n // 2)
        if n <= 1: break
    return total

In [ ]:
# ── Cell 7: Loss + accuracy + AQCA metrics ───────────────
def cross_entropy(labels, probs):
    loss = 0.0
    for l, p in zip(labels, probs):
        p = pnp.clip(p, 1e-9, 1.0)
        loss -= l * pnp.log(p[0]) + (1-l) * pnp.log(p[1])
    return loss / len(labels)

def accuracy(labels, probs):
    preds = [1 if p[0] > p[1] else 0 for p in probs]
    return np.mean(np.array(preds) == np.array(labels))

def compute_gvs(grad_history):
    if len(grad_history) < 2: return 0.0
    G = np.stack(grad_history, axis=0)
    return float(np.mean(np.var(G, axis=0)))

def compute_fmf(circuit, params, X, Y):
    probs = [circuit(x, params) for x in X[:20]]
    fid, count = 0.0, 0
    for i in range(len(probs)):
        for j in range(i+1, len(probs)):
            if Y[i] == Y[j]:
                pi = np.array(probs[i]); pj = np.array(probs[j])
                fid += float(np.dot(pi,pj) /
                             (np.linalg.norm(pi)*np.linalg.norm(pj)+1e-9))
                count += 1
    return fid / max(count, 1)

In [ ]:
# ── Cell 8: Training loop ────────────────────────────────
def train(X_tr, Y_tr, X_te, Y_te,
          noise_model=None, use_aqca=True, fixed_depth=2):

    depth   = fixed_depth
    circuit = build_circuit(depth, noise_model)
    params  = pnp.array(np.random.randn(n_params(depth)), requires_grad=True)
    opt     = qml.NesterovMomentumOptimizer(stepsize=LR)
    grad_hist, loss_hist, adapt_log = [], [], []

    for step in range(STEPS):
        idx = np.random.randint(0, len(X_tr), BATCH)
        Xb  = [X_tr[i] for i in idx]
        Yb  = [Y_tr[i] for i in idx]

        def cost(p):
            return cross_entropy(Yb, [circuit(x, p) for x in Xb])

        params, loss = opt.step_and_cost(cost, params)
        loss_hist.append(float(loss))

        if use_aqca:
            g = qml.grad(cost)(params)
            grad_hist.append(np.array(g))
            if len(grad_hist) > W_WINDOW:
                grad_hist.pop(0)

        if use_aqca and step > 0 and step % K_ADAPT == 0:
            gvs = compute_gvs(grad_hist)
            fmf = compute_fmf(circuit, params, Xb, Yb)
            action = None
            if gvs < GVS_LO and fmf < FMF_LO and depth > D_MIN:
                depth -= 1;  action = f"depth↓→{depth}"
            elif gvs > GVS_HI and fmf < FMF_HI and depth < D_MAX:
                depth += 1;  action = f"depth↑→{depth}"
            if action:
                adapt_log.append({'step': step, 'action': action})
                P_new = n_params(depth)
                if P_new > len(params):
                    extra  = pnp.array(np.random.randn(P_new-len(params))*0.01,
                                       requires_grad=True)
                    params = pnp.concatenate([params, extra])
                else:
                    params = params[:P_new]
                circuit = build_circuit(depth, noise_model)
                print(f"  Step {step}: {action}  GVS={gvs:.2e}  FMF={fmf:.3f}")

        if step % 20 == 0:
            acc = accuracy(Y_te[:30],
                           [circuit(x, params) for x in X_te[:30]])
            print(f"  step {step:3d}  loss={loss:.4f}  acc={acc:.3f}")

    final_acc = accuracy(Y_te, [circuit(x, params) for x in X_te])
    return final_acc, loss_hist, adapt_log

In [ ]:
# ── Cell 9: Run experiment ───────────────────────────────
np.random.seed(42)
results = {}

for p in NOISE_LEVELS:
    print(f"\n{'='*50}  p={p}")
    nm = build_noise_model(p_depol=p, p_damp=p/2,
                           readout_err=0.03 if p > 0 else 0.0)

    print("  [Fixed depth d=2]")
    acc_f, _, _ = train(X_train, Y_train, X_test, Y_test,
                        noise_model=nm, use_aqca=False, fixed_depth=2)

    print("  [AQCA]")
    acc_a, lh, al = train(X_train, Y_train, X_test, Y_test,
                          noise_model=nm, use_aqca=True, fixed_depth=2)

    results[p] = {'fixed': round(acc_f*100,1),
                  'aqca':  round(acc_a*100,1),
                  'gain':  round((acc_a-acc_f)*100,1),
                  'adapts': len(al)}
    print(f"  Fixed={results[p]['fixed']}%  "
          f"AQCA={results[p]['aqca']}%  "
          f"Gain=+{results[p]['gain']}%")


==================================================  p=0.0
  [Fixed depth d=2]
  step   0  loss=0.7283  acc=0.333


In [ ]:
# ── Cell 10: Print summary + save ────────────────────────
print(f"\n{'p':>6}  {'Fixed%':>7}  {'AQCA%':>6}  {'Gain':>5}  {'Adapts':>6}")
for p, r in results.items():
    print(f"{p:>6.3f}  {r['fixed']:>7.1f}  "
          f"{r['aqca']:>6.1f}  {r['gain']:>+5.1f}  {r['adapts']:>6}")

with open("ibm_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved to ibm_results.json")